# Track 1: Multi-View 3D Semantic Occupancy Prediction
Individual, non-AI-assisted work. Do not use Gemini or AI completion. This notebook provides setup and execution, not solutions.

Release: November 4, 2026. Due: December 9, 2026, 23:59 America/Chicago.

Save your own notebook copy in Drive. Select Runtime > Change runtime type > GPU. Free GPU availability/runtime duration is not guaranteed. Run the numbered sections in order. For full instructions see SETUP.md.

Saving this notebook alone does NOT save edits to Python files, weights or outputs. The persistent project below saves these separately to Drive; datasets stay on local runtime disk.

## 1. Mount Drive and reuse your project
Rerun sections 1-4 after a runtime reset. Never delete/reset the persistent project just to reconnect. Change PROJECT if you want a separate experiment. Google requests consent to mount your Drive.

In [ ]:
import os, sys, shutil, subprocess, tempfile, zipfile, stat
from pathlib import Path
if not ((3, 11) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError('Select a Python 3.11/3.12 runtime; this runtime has not been validated.')
from google.colab import drive
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/cs498-track1')
DATA = Path('/content/track1-data')
os.environ['HF_HOME'] = '/content/track1-hf-cache'
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
print('Persistent project:', PROJECT, '\nEphemeral dataset:', DATA)
def staged_upload():
    # Colab writes uploaded basenames to cwd before returning. Stage away from saved work.
    from google.colab import files
    previous = Path.cwd()
    try:
        with tempfile.TemporaryDirectory() as staging:
            os.chdir(staging)
            return files.upload()
    finally:
        os.chdir(previous)


## 2. Obtain the release (first session only)
Choose github after the course publishes the release, or upload for a course-provided final-track1.zip. An existing project is reused without overwriting any edited files. Only upload the official starter, never a staff solution. A pre-release missing directory is a publication issue, not an environment error.

In [ ]:
STARTER_SOURCE = 'github'  # change to 'upload' for the official release ZIP
if (PROJECT / 'run.py').is_file():
    print('Reusing existing project; no files overwritten.')
elif PROJECT.exists():
    raise RuntimeError('Project exists but run.py is missing. Inspect it or choose a new PROJECT; do not overwrite your work.')
else:
    with tempfile.TemporaryDirectory() as temporary:
        root = Path(temporary)
        if STARTER_SOURCE == 'github':
            checkout = root / 'course'
            subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', 'https://github.com/shenlongwang/cs498-fall2026.git', str(checkout)], check=True)
            subprocess.run(['git', '-C', str(checkout), 'sparse-checkout', 'set', 'final-track1'], check=True)
            starter = checkout / 'final-track1'
        elif STARTER_SOURCE == 'upload':
            from google.colab import files
            uploaded = staged_upload()
            if list(uploaded) != ['final-track1.zip']:
                raise ValueError('Choose exactly the course-provided final-track1.zip.')
            import io
            with zipfile.ZipFile(io.BytesIO(uploaded['final-track1.zip'])) as archive:
                for member in archive.infolist():
                    target = (root / member.filename).resolve()
                    if not target.is_relative_to(root.resolve()) or stat.S_ISLNK(member.external_attr >> 16):
                        raise ValueError('Unsafe ZIP path/link')
                archive.extractall(root)
            starter = root / 'final-track1'
        else:
            raise ValueError('Choose github or upload')
        if not (starter / 'run.py').is_file():
            raise RuntimeError('Current student release is not published at this source yet. Use the official ZIP or retry after release.')
        PROJECT.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(starter, PROJECT, ignore=shutil.ignore_patterns('.git', '__pycache__'))
os.chdir(PROJECT)
print('Working directory:', Path.cwd())


## 3. Install this runtime's tools
Repeat after reset. Preserve compatible preinstalled Torch rather than replacing it alone and breaking torchvision/torchaudio. If installation requests a runtime restart, restart and rerun sections 1-4. LaTeX installation is for the report; it may take a few minutes.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'student_submission/requirements.txt'], check=True)
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'latexmk', 'texlive-latex-base', 'texlive-latex-recommended', 'texlive-latex-extra'], check=True)
def run(*arguments):
    subprocess.run([sys.executable, 'run.py', *map(str, arguments)], cwd=PROJECT, check=True)
run('doctor', '--require-gpu', '--require-latex', '--data-root', DATA, '--min-free-gib', '15', '--output', 'outputs/environment.json')
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'], check=True)


## 4. Anonymous subset download and smoke
No HF login/token is needed. Local data/cache disappear after reset and must be downloaded again. The eight untouched TODOs yield XFAILs and noncompetitive output. Smoke checks do not establish correctness or completed-model GPU memory. Final weights are not overwritten by smoke.

In [ ]:
run('download', '--split', 'train', '--houses', 'train_house_000', '--output', DATA)
run('download', '--split', 'val', '--houses', 'val_house_000', '--output', DATA)
run('train', '--smoke', '--device', 'cuda', '--batch-size', '8', '--workers', '0', '--data-root', DATA)


## 5. Implement your method yourself
Open the Files panel and navigate to drive/MyDrive/cs498-track1/student_submission/semocc/. Edit model.py and learning.py and save the actual files, or edit locally and use the optional upload below. The README/PDF list the eight functions.

Alternative methods use the same adapter contract. Declare their inference dependencies in student_submission/requirements.txt and rerun installation after changing it. Section 6 is the baseline trainer only: replacement methods run their own training command in an added cell (keep code/weights in student_submission/), then rejoin section 7. Unused baseline-specific tests do not apply; see README for common protocol/workflow tests.

Do not confuse notebook cells with .py files used by the CLI. Re-run focused tests after saving. Complete method.json before final evidence. Saving to Git alone will not save ignored checkpoints/outputs.

In [ ]:
UPLOAD_EDITED_FILE = False  # explicitly enable only to replace one chosen file
TARGET = Path('student_submission/semocc/model.py')  # or learning.py / your own method file
if UPLOAD_EDITED_FILE:
    from google.colab import files
    import time, hashlib
    if TARGET.is_absolute() or '..' in TARGET.parts or TARGET.parts[0] != 'student_submission':
        raise ValueError('Choose a relative student_submission file')
    uploaded = staged_upload()
    if list(uploaded) != [TARGET.name]:
        raise ValueError('Upload exactly the selected filename')
    if TARGET.exists():
        backup = Path('outputs/editor_backups') / str(time.time_ns()) / TARGET.name
        backup.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(TARGET, backup)
    TARGET.parent.mkdir(parents=True, exist_ok=True)
    TARGET.write_bytes(uploaded[TARGET.name])
    print('Saved', TARGET, 'SHA256', hashlib.sha256(TARGET.read_bytes()).hexdigest())
else:
    print('No files changed. Edit/save the .py files in Files or opt into upload.')


## 6. Full-data training or recovery
Enable only after implementing/testing your method. The first full download obtains all 200 Train and 10 Val houses. On reconnect rerun setup/download with the same pinned data; persistent checkpoints stay in Drive. An unfinished epoch is repeated. Resume requires unchanged code, Torch version, device, batch size and total epoch target. Changed algorithms/settings are new experiments, not exact resumes.

This uses batch8/workers0 as a starting setting, not a guarantee for every allocated GPU. If memory fails, start a new smaller-batch run. No fixed free-runtime lifetime or T4 training duration is promised.

In [ ]:
RUN_FULL_TRAINING = False
TRAIN_OUTPUT = Path('outputs/train')  # use outputs/experiment2 for a new algorithm/settings
RESUME_IF_AVAILABLE = True  # False starts a new experiment in a fresh TRAIN_OUTPUT
if RUN_FULL_TRAINING:
    run('doctor', '--require-gpu')
    run('download', '--split', 'train', '--output', DATA)
    run('download', '--split', 'val', '--output', DATA)
    run('doctor', '--require-gpu', '--data-root', DATA, '--splits', 'train', 'val')
    command = ['train', '--device', 'cuda', '--epochs', '15', '--batch-size', '8', '--workers', '0', '--data-root', DATA, '--output', TRAIN_OUTPUT]
    recovery = TRAIN_OUTPUT / 'checkpoint_last.pt'
    if recovery.is_file():
        if not RESUME_IF_AVAILABLE:
            raise RuntimeError('Choose a fresh TRAIN_OUTPUT for a new experiment; existing state is not overwritten.')
        command += ['--resume', recovery]
    run(*command)
else:
    print('Skipped full training. Complete your method and enable this cell when ready.')


## 7. Final evidence and CSV
Use your own final Val-selected method. First complete student_submission/method.json. Run complete Val/Test, not partial smoke. After editing code/weights, regenerate evidence. Fill report_template.tex yourself with honest analysis before building the report. The provided plots work for any canonical-grid method.

In [ ]:
GENERATE_FINAL_EVIDENCE = False
if GENERATE_FINAL_EVIDENCE:
    run('download', '--split', 'val', '--output', DATA)
    run('download', '--split', 'test', '--output', DATA)
    run('evaluate', '--data-root', DATA)
    run('figures', '--data-root', DATA)
    run('predict', '--data-root', DATA)
    run('submit', 'outputs/kaggle/submission.csv', '--data-root', DATA)  # local validation only
else:
    print('Skipped final evidence. Do not submit placeholder smoke predictions.')


### Download the CSV before writing your Kaggle result into the report
This is independent of report compilation and LMS packaging. Enable only after full Test inference. Submit the downloaded CSV in your own Kaggle browser session, then record its completed score/entry in your report.

In [ ]:
DOWNLOAD_KAGGLE_CSV = False
if DOWNLOAD_KAGGLE_CSV:
    run('submit', 'outputs/kaggle/submission.csv', '--data-root', DATA)  # validate; do not upload
    from google.colab import files
    files.download('outputs/kaggle/submission.csv')
else:
    print('Enable after final inference to download your CSV for Kaggle.')


## 8. Kaggle and LMS submission
Download submission.csv and upload it in your own browser after signing into Kaggle, joining the competition and accepting the rules. Competition: https://www.kaggle.com/competitions/cs498-fall-2026-multiview-occupancy . It is private: first open the course invitation https://www.kaggle.com/t/f04e3dce46f047c5b52deec9fbe73701 while signed in to a Kaggle account with a verified @illinois.edu email (until then the page shows 404). No Kaggle token is needed for browser upload. Never put credentials in notebook cells or Drive files.

After writing your report, choose REPORT_SOURCE: existing preserves report.pdf already in the project; latex compiles your completed report_template.tex and replaces report.pdf; upload accepts your externally compiled report.pdf. Packaging never silently recompiles an external report. Set the two IDs below and enable packaging. Download the ZIP AND package_manifest.json for LMS. If multiple downloads are blocked, use the Files panel individually. Verify the files actually reached your computer and keep the exact CSV.

In [ ]:
PACKAGE_FINAL = False
NETID = 'YOUR_NETID'
KAGGLE_USERNAME = 'YOUR_KAGGLE_NAME'
REPORT_SOURCE = 'existing'  # existing, latex, or upload; never silently replace an external report
if PACKAGE_FINAL:
    if NETID == 'YOUR_NETID' or KAGGLE_USERNAME == 'YOUR_KAGGLE_NAME':
        raise ValueError('Set your real IDs and complete method.json/report first')
    from google.colab import files
    if REPORT_SOURCE == 'latex':
        run('report')
    elif REPORT_SOURCE == 'upload':
        uploaded = staged_upload()
        if list(uploaded) != ['report.pdf']:
            raise ValueError('Choose exactly your completed report.pdf')
        Path('report.pdf').write_bytes(uploaded['report.pdf'])
    elif REPORT_SOURCE != 'existing':
        raise ValueError('Choose existing, latex or upload')
    if not Path('report.pdf').is_file():
        raise FileNotFoundError('Provide report.pdf or explicitly choose latex/upload above')
    run('package', '--student-id', NETID, '--kaggle-username', KAGGLE_USERNAME, '--data-root', DATA)
    for path in [Path('outputs/kaggle/submission.csv'), Path(f'outputs/submission_package/cs498-track1-{NETID}.zip'), Path('outputs/submission_package/package_manifest.json')]:
        files.download(str(path))
else:
    print('Skipped packaging. Complete the report, method declaration and final evidence first.')
